# M1 — kiểm tra SFT (8 bước)

Chạy thử trên 32 câu train để kiểm tra train, lưu adapter và nạp lại khi suy luận. Chưa dùng kết quả này để kết luận chất lượng mô hình. Đây là run lịch sử: chỉ chạy ô khởi tạo và ô xác minh cuối nếu muốn kiểm tra lại; đừng Chạy tất cả vì sẽ tạo run/train mới.

In [2]:
import hashlib
import json
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU: chỉ xác minh run cũ")

Mounted at /content/drive
Tesla T4


In [ ]:
assert torch.cuda.is_available(), "Cần GPU runtime để train; xác minh run cũ bằng ô cuối"
DRIVE_ROOT = Path("/content/drive/MyDrive/Vietnamese_VQA_NLE")
RUN_ID = "M1_smoke_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN = DRIVE_ROOT / "runs" / RUN_ID
RUN.mkdir(parents=True, exist_ok=False)
SOURCE = DRIVE_ROOT / "source/project_source.zip"
ARCHIVE = DRIVE_ROOT / "datasets/images/coco_smoke_500.zip"
ANNOTATIONS = DRIVE_ROOT / "datasets/processed/smoke_500.jsonl"
assert (
    hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
    == "4e0fce747dee9a1beaf36e2617deae6582115020d7c219f315525e329a4024f9"
)
LOCAL = Path("/content") / RUN_ID
LOCAL.mkdir()


def unpack(archive, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        assert all((destination / n).resolve().is_relative_to(destination) for n in z.namelist())
        z.extractall(destination)


unpack(SOURCE, LOCAL)
REPO = LOCAL / "Vietnamese-VQA-Explanation"
assert (REPO / "pyproject.toml").is_file(), list(LOCAL.iterdir())
OVERLAY = json.loads(
    '{"scripts/run_m0.py":"from __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport random\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom vqa_explanation.inference.prompting import parse_prediction\\nfrom vqa_explanation.inference.qwen25_vl import Qwen25VLRunner\\n\\nDEFAULT_MODEL = \\"Qwen/Qwen2.5-VL-3B-Instruct\\"\\nDEFAULT_REVISION = \\"66285546d2b821cf421d4f5eb2576359d3770cd3\\"\\n\\n\\ndef read_jsonl(path: Path) -> list[dict[str, Any]]:\\n    return [json.loads(line) for line in path.read_text(encoding=\\"utf-8\\").splitlines() if line]\\n\\n\\ndef select_records(\\n    records: list[dict[str, Any]], split: str, limit: int, seed: int\\n) -> list[dict[str, Any]]:\\n    selected = [record for record in records if record.get(\\"split\\") == split]\\n    random.Random(seed).shuffle(selected)\\n    if limit > 0:\\n        selected = selected[:limit]\\n    return selected\\n\\n\\ndef parse_args() -> argparse.Namespace:\\n    parser = argparse.ArgumentParser(description=\\"Run the frozen M0 pretrained baseline.\\")\\n    parser.add_argument(\\"--annotations\\", type=Path, required=True)\\n    parser.add_argument(\\"--image-root\\", type=Path, required=True)\\n    parser.add_argument(\\"--output\\", type=Path, required=True)\\n    parser.add_argument(\\"--split\\", choices=(\\"validation\\", \\"test\\"), default=\\"validation\\")\\n    parser.add_argument(\\"--limit\\", type=int, default=20)\\n    parser.add_argument(\\"--seed\\", type=int, default=20260918)\\n    parser.add_argument(\\"--model-id\\", default=DEFAULT_MODEL)\\n    parser.add_argument(\\"--revision\\", default=DEFAULT_REVISION)\\n    parser.add_argument(\\"--max-new-tokens\\", type=int, default=128)\\n    parser.add_argument(\\"--load-in-4bit\\", action=\\"store_true\\")\\n    parser.add_argument(\\"--adapter\\", type=Path)\\n    parser.add_argument(\\"--experiment-id\\", default=\\"M0_PRETRAINED\\")\\n    parser.add_argument(\\"--max-pixels\\", type=int, default=1024 * 28 * 28)\\n    return parser.parse_args()\\n\\n\\ndef main() -> None:\\n    args = parse_args()\\n    if args.output.exists():\\n        raise SystemExit(f\\"Output exists; choose a new path: {args.output}\\")\\n    if args.adapter and args.experiment_id == \\"M0_PRETRAINED\\":\\n        raise SystemExit(\\"An adapter run must have a distinct --experiment-id\\")\\n    records = select_records(read_jsonl(args.annotations), args.split, args.limit, args.seed)\\n    if not records:\\n        raise SystemExit(f\\"No records found for split={args.split!r}\\")\\n\\n    missing = [\\n        str(args.image_root / record[\\"image_path\\"])\\n        for record in records\\n        if not (args.image_root / record[\\"image_path\\"]).is_file()\\n    ]\\n    if missing:\\n        raise SystemExit(f\\"Missing {len(missing)} selected images; first: {missing[0]}\\")\\n\\n    runner = Qwen25VLRunner(\\n        args.model_id,\\n        args.revision,\\n        load_in_4bit=args.load_in_4bit,\\n        adapter_path=str(args.adapter) if args.adapter else None,\\n        max_pixels=args.max_pixels,\\n    )\\n    args.output.parent.mkdir(parents=True, exist_ok=True)\\n    with args.output.open(\\"w\\", encoding=\\"utf-8\\") as destination:\\n        for index, record in enumerate(records, start=1):\\n            started = time.perf_counter()\\n            raw = runner.generate(\\n                args.image_root / record[\\"image_path\\"],\\n                record[\\"question\\"],\\n                max_new_tokens=args.max_new_tokens,\\n            )\\n            parsed = parse_prediction(raw)\\n            result = {\\n                \\"experiment_id\\": args.experiment_id,\\n                \\"adapter\\": str(args.adapter) if args.adapter else None,\\n                \\"max_pixels\\": args.max_pixels,\\n                \\"model_id\\": args.model_id,\\n                \\"model_revision\\": args.revision,\\n                \\"seed\\": args.seed,\\n                \\"question_id\\": record[\\"question_id\\"],\\n                \\"image_id\\": record[\\"image_id\\"],\\n                \\"image_path\\": record[\\"image_path\\"],\\n                \\"split\\": record[\\"split\\"],\\n                \\"question\\": record[\\"question\\"],\\n                \\"reference_answer\\": record[\\"answer\\"],\\n                \\"reference_explanations\\": record[\\"explanations\\"],\\n                \\"prediction_answer\\": parsed.answer,\\n                \\"prediction_explanation\\": parsed.explanation,\\n                \\"format_valid\\": parsed.format_valid,\\n                \\"raw_output\\": raw,\\n                \\"latency_seconds\\": round(time.perf_counter() - started, 4),\\n            }\\n            destination.write(json.dumps(result, ensure_ascii=False) + \\"\\\\n\\")\\n            destination.flush()\\n            print(f\\"[{index}/{len(records)}] {record[\'question_id\']} format={parsed.format_valid}\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n","src/vqa_explanation/training/sft_data.py":"\\"\\"\\"Data and assistant-only supervision for image-conditioned SFT.\\"\\"\\"\\n\\nfrom pathlib import Path\\n\\nfrom PIL import Image\\n\\nfrom vqa_explanation.data.formatting import format_target\\nfrom vqa_explanation.data.schema import VQASample\\nfrom vqa_explanation.inference.prompting import SYSTEM_PROMPT, build_user_prompt\\n\\n\\ndef select_training_records(records: list[dict], limit: int, seed: int) -> list[dict]:\\n    import random\\n\\n    selected = []\\n    seen = set()\\n    heldout_images = {str(r[\\"image_id\\"]) for r in records if r[\\"split\\"] != \\"train\\"}\\n    for raw in records:\\n        if raw[\\"split\\"] != \\"train\\":\\n            continue\\n        sample = VQASample.from_dict(raw)\\n        sample.validate_ids()\\n        if sample.question_id in seen:\\n            raise ValueError(f\\"Duplicate training question: {sample.question_id}\\")\\n        if sample.image_id in heldout_images:\\n            raise ValueError(f\\"Train/held-out image overlap: {sample.image_id}\\")\\n        seen.add(sample.question_id)\\n        selected.append(raw)\\n    random.Random(seed).shuffle(selected)\\n    if limit > 0:\\n        selected = selected[:limit]\\n    if not selected:\\n        raise ValueError(\\"No training samples\\")\\n    return selected\\n\\n\\ndef messages_for(record: dict) -> list[dict]:\\n    return [\\n        {\\"role\\": \\"system\\", \\"content\\": [{\\"type\\": \\"text\\", \\"text\\": SYSTEM_PROMPT}]},\\n        {\\n            \\"role\\": \\"user\\",\\n            \\"content\\": [\\n                {\\"type\\": \\"image\\"},\\n                {\\"type\\": \\"text\\", \\"text\\": build_user_prompt(record[\\"question\\"])},\\n            ],\\n        },\\n    ]\\n\\n\\ndef assistant_labels(full: list[int], prompt: list[int], image_ids: set[int]) -> list[int]:\\n    \\"\\"\\"Refuse template mismatches instead of accidentally training on the question.\\"\\"\\"\\n    if not prompt or full[: len(prompt)] != prompt:\\n        raise ValueError(\\"Prompt is not an exact token prefix of the full conversation\\")\\n    if len(full) <= len(prompt):\\n        raise ValueError(\\"No assistant tokens to supervise\\")\\n    labels = [-100] * len(prompt) + full[len(prompt) :]\\n    labels = [\\n        -100 if token in image_ids else label for token, label in zip(full, labels, strict=True)\\n    ]\\n    if all(label == -100 for label in labels):\\n        raise ValueError(\\"All assistant labels are masked\\")\\n    return labels\\n\\n\\nclass VQACollator:\\n    \\"\\"\\"Batch size one avoids ambiguous image grids/padding on the first GPU smoke run.\\"\\"\\"\\n\\n    def __init__(self, processor, image_root: Path, max_length: int = 1536):\\n        self.processor = processor\\n        self.image_root = image_root.resolve()\\n        self.max_length = max_length\\n\\n    def __call__(self, records):\\n        import torch\\n\\n        if len(records) != 1:\\n            raise ValueError(\\"Use batch size 1 and gradient accumulation\\")\\n        record = records[0]\\n        path = (self.image_root / record[\\"image_path\\"]).resolve()\\n        if not path.is_relative_to(self.image_root):\\n            raise ValueError(\\"Image path escapes image root\\")\\n        with Image.open(path) as opened:\\n            image = opened.convert(\\"RGB\\")\\n        messages = messages_for(record)\\n        prompt = self.processor.apply_chat_template(\\n            messages,\\n            tokenize=False,\\n            add_generation_prompt=True,\\n        )\\n        # One reference per question avoids overweighting multi-reference questions.\\n        target = format_target(record[\\"answer\\"], record[\\"explanations\\"][0])\\n        full = self.processor.apply_chat_template(\\n            messages + [{\\"role\\": \\"assistant\\", \\"content\\": [{\\"type\\": \\"text\\", \\"text\\": target}]}],\\n            tokenize=False,\\n            add_generation_prompt=False,\\n        )\\n        batch = self.processor(text=[full], images=[image], return_tensors=\\"pt\\")\\n        prompt_batch = self.processor(text=[prompt], images=[image], return_tensors=\\"pt\\")\\n        full_ids = batch[\\"input_ids\\"][0].tolist()\\n        prompt_ids = prompt_batch[\\"input_ids\\"][0].tolist()\\n        if len(full_ids) > self.max_length:\\n            raise ValueError(\\n                f\\"{record[\'question_id\']}: {len(full_ids)} tokens exceeds {self.max_length}; \\"\\n                \\"reduce image resolution or raise max_length; do not truncate image tokens\\"\\n            )\\n        vocab = self.processor.tokenizer.get_vocab()\\n        image_ids = {\\n            vocab[t]\\n            for t in (\\n                \\"<|image_pad|>\\",\\n                \\"<|video_pad|>\\",\\n                \\"<|vision_start|>\\",\\n                \\"<|vision_end|>\\",\\n            )\\n            if t in vocab\\n        }\\n        labels = assistant_labels(full_ids, prompt_ids, image_ids)\\n        batch[\\"labels\\"] = torch.tensor([labels], dtype=torch.long)\\n        return batch\\n","configs/training/sft_smoke.yaml":"# Engineering check only; not the final M1 model.\\nseed: 20260918\\nlimit: 32\\nmax_steps: 8\\ngrad_accum: 2\\nlearning_rate: 0.00002\\nsave_steps: 4\\nmin_pixels: 200704\\nmax_pixels: 200704\\nmax_length: 1536\\n","scripts/evaluate.py":"\\"\\"\\"Evaluate JSONL predictions produced by the VQA inference scripts.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport math\\nfrom pathlib import Path\\nfrom statistics import mean, median\\nfrom typing import Any\\n\\nfrom vqa_explanation.evaluation.answer_metrics import exact_match, token_f1\\n\\n\\ndef read_predictions(path: Path) -> list[dict[str, Any]]:\\n    \\"\\"\\"Read prediction records from JSONL, reporting malformed rows clearly.\\"\\"\\"\\n    records: list[dict[str, Any]] = []\\n    with path.open(encoding=\\"utf-8\\") as source:\\n        for line_number, line in enumerate(source, start=1):\\n            if not line.strip():\\n                continue\\n            try:\\n                record = json.loads(line)\\n            except json.JSONDecodeError as error:\\n                message = f\\"{path}: invalid JSON on line {line_number}: {error.msg}\\"\\n                raise ValueError(message) from error\\n            if not isinstance(record, dict):\\n                raise ValueError(f\\"{path}: line {line_number} must contain a JSON object\\")\\n            records.append(record)\\n    if not records:\\n        raise ValueError(f\\"No prediction records found in {path}\\")\\n    return records\\n\\n\\ndef evaluate_predictions(records: list[dict[str, Any]]) -> dict[str, Any]:\\n    \\"\\"\\"Calculate answer accuracy, output-format rate, and inference latency.\\"\\"\\"\\n    if not records:\\n        raise ValueError(\\"No prediction records to evaluate\\")\\n\\n    answer_scores: list[float] = []\\n    f1_scores: list[float] = []\\n    format_values: list[float] = []\\n    latencies: list[float] = []\\n\\n    for index, record in enumerate(records, start=1):\\n        for field in (\\"prediction_answer\\", \\"reference_answer\\"):\\n            if not isinstance(record.get(field), str):\\n                raise ValueError(f\\"Record {index} must contain string field {field!r}\\")\\n        answer_scores.append(exact_match(record[\\"prediction_answer\\"], record[\\"reference_answer\\"]))\\n        f1_scores.append(token_f1(record[\\"prediction_answer\\"], record[\\"reference_answer\\"]))\\n\\n        if \\"format_valid\\" in record:\\n            format_values.append(float(bool(record[\\"format_valid\\"])))\\n        if record.get(\\"latency_seconds\\") is not None:\\n            try:\\n                latency = float(record[\\"latency_seconds\\"])\\n            except (TypeError, ValueError) as error:\\n                raise ValueError(f\\"Record {index} has an invalid latency_seconds value\\") from error\\n            if not math.isfinite(latency) or latency < 0:\\n                raise ValueError(f\\"Record {index} latency_seconds must be finite and non-negative\\")\\n            latencies.append(latency)\\n\\n    return {\\n        \\"n\\": len(records),\\n        \\"exact_match\\": mean(answer_scores),\\n        \\"token_f1\\": mean(f1_scores),\\n        \\"format_valid_n\\": len(format_values),\\n        \\"format_rate\\": mean(format_values) if format_values else None,\\n        \\"latency_n\\": len(latencies),\\n        \\"mean_latency_seconds\\": mean(latencies) if latencies else None,\\n        \\"median_latency_seconds\\": median(latencies) if latencies else None,\\n    }\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"predictions\\", type=Path, help=\\"Input JSONL prediction file\\")\\n    parser.add_argument(\\"--split\\", choices=(\\"train\\", \\"validation\\", \\"test\\"))\\n    parser.add_argument(\\"--output\\", type=Path, help=\\"Optional path for the JSON metrics report\\")\\n    args = parser.parse_args()\\n\\n    try:\\n        records = read_predictions(args.predictions)\\n        if args.split:\\n            records = [record for record in records if record.get(\\"split\\") == args.split]\\n            if not records:\\n                raise ValueError(f\\"No prediction records found for split {args.split!r}\\")\\n        metrics = evaluate_predictions(records)\\n    except (OSError, ValueError) as error:\\n        parser.error(str(error))\\n\\n    metrics[\\"split\\"] = args.split or \\"all\\"\\n    output = json.dumps(metrics, ensure_ascii=False, indent=2)\\n    if args.output:\\n        args.output.parent.mkdir(parents=True, exist_ok=True)\\n        args.output.write_text(output + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    print(output)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n","src/vqa_explanation/inference/qwen25_vl.py":"from __future__ import annotations\\n\\nfrom pathlib import Path\\n\\nfrom PIL import Image\\n\\nfrom vqa_explanation.inference.prompting import SYSTEM_PROMPT, build_user_prompt\\n\\n\\nclass Qwen25VLRunner:\\n    \\"\\"\\"Thin inference adapter; heavyweight imports stay optional for local tooling.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        model_id: str,\\n        revision: str,\\n        *,\\n        load_in_4bit: bool = False,\\n        min_pixels: int = 256 * 28 * 28,\\n        max_pixels: int = 1024 * 28 * 28,\\n        adapter_path: str | None = None,\\n    ) -> None:\\n        try:\\n            import torch\\n            from transformers import (\\n                AutoProcessor,\\n                BitsAndBytesConfig,\\n                Qwen2_5_VLForConditionalGeneration,\\n            )\\n        except ImportError as exc:  # pragma: no cover - exercised on GPU runtime\\n            raise RuntimeError(\\n                \'Install the training dependencies with: pip install -e \\".[train]\\"\'\\n            ) from exc\\n\\n        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\\n        quantization_config = None\\n        if load_in_4bit:\\n            quantization_config = BitsAndBytesConfig(\\n                load_in_4bit=True,\\n                bnb_4bit_quant_type=\\"nf4\\",\\n                bnb_4bit_compute_dtype=dtype,\\n            )\\n\\n        self.processor = AutoProcessor.from_pretrained(\\n            model_id,\\n            revision=revision,\\n            min_pixels=min_pixels,\\n            max_pixels=max_pixels,\\n        )\\n        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\\n            model_id,\\n            revision=revision,\\n            device_map=\\"auto\\",\\n            torch_dtype=dtype,\\n            quantization_config=quantization_config,\\n            attn_implementation=\\"sdpa\\",\\n        )\\n        if adapter_path:\\n            from peft import PeftModel\\n\\n            self.model = PeftModel.from_pretrained(self.model, adapter_path, is_trainable=False)\\n        self.model.config.use_cache = True\\n        self.model.eval()\\n        self.torch = torch\\n\\n    def generate(self, image_path: Path, question: str, *, max_new_tokens: int = 128) -> str:\\n        messages = [\\n            {\\"role\\": \\"system\\", \\"content\\": [{\\"type\\": \\"text\\", \\"text\\": SYSTEM_PROMPT}]},\\n            {\\n                \\"role\\": \\"user\\",\\n                \\"content\\": [\\n                    {\\"type\\": \\"image\\"},\\n                    {\\"type\\": \\"text\\", \\"text\\": build_user_prompt(question)},\\n                ],\\n            },\\n        ]\\n        prompt = self.processor.apply_chat_template(\\n            messages,\\n            add_generation_prompt=True,\\n            tokenize=False,\\n        )\\n        with Image.open(image_path) as opened:\\n            image = opened.convert(\\"RGB\\")\\n        inputs = self.processor(text=[prompt], images=[image], return_tensors=\\"pt\\")\\n        inputs = inputs.to(self.model.device)\\n        with self.torch.inference_mode():\\n            generated = self.model.generate(\\n                **inputs,\\n                do_sample=False,\\n                max_new_tokens=max_new_tokens,\\n            )\\n        prompt_length = inputs[\\"input_ids\\"].shape[1]\\n        completion_ids = generated[:, prompt_length:]\\n        return self.processor.batch_decode(\\n            completion_ids,\\n            skip_special_tokens=True,\\n            clean_up_tokenization_spaces=False,\\n        )[0].strip()\\n","src/vqa_explanation/training/sft.py":"\\"\\"\\"Single-GPU QLoRA training with explicit smoke-test acceptance checks.\\"\\"\\"\\n\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport math\\nimport subprocess\\nfrom pathlib import Path\\n\\nimport yaml\\n\\nfrom vqa_explanation.data.io import read_jsonl, write_json\\nfrom vqa_explanation.training.sft_data import VQACollator, select_training_records\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--annotations\\", type=Path, required=True)\\n    parser.add_argument(\\"--image-root\\", type=Path, required=True)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--config\\", type=Path, default=Path(\\"configs/training/sft_smoke.yaml\\"))\\n    parser.add_argument(\\n        \\"--model-config\\",\\n        type=Path,\\n        default=Path(\\"configs/model/qwen25_vl_3b.yaml\\"),\\n    )\\n    parser.add_argument(\\"--preflight-only\\", action=\\"store_true\\")\\n    args = parser.parse_args()\\n    cfg = yaml.safe_load(args.config.read_text(encoding=\\"utf-8\\"))\\n    model_cfg = yaml.safe_load(args.model_config.read_text(encoding=\\"utf-8\\"))\\n    records = select_training_records(\\n        list(read_jsonl(args.annotations)),\\n        cfg[\\"limit\\"],\\n        cfg[\\"seed\\"],\\n    )\\n    root = args.image_root.resolve()\\n    for record in records:\\n        path = (root / record[\\"image_path\\"]).resolve()\\n        if not path.is_relative_to(root) or not path.is_file():\\n            raise ValueError(f\\"Missing or unsafe image: {record[\'image_path\']}\\")\\n    if args.preflight_only:\\n        print(f\\"Data preflight passed: {len(records)} train questions; images present\\")\\n        return\\n    if args.output_dir.exists() and any(args.output_dir.iterdir()):\\n        raise ValueError(\\"Output directory is not empty; choose a new run directory\\")\\n\\n    import torch\\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\\n    from transformers import (\\n        AutoProcessor,\\n        BitsAndBytesConfig,\\n        Qwen2_5_VLForConditionalGeneration,\\n        Trainer,\\n        TrainingArguments,\\n        set_seed,\\n    )\\n\\n    if not torch.cuda.is_available():\\n        raise RuntimeError(\\"This entry point requires a CUDA GPU; use Colab\\")\\n    set_seed(cfg[\\"seed\\"])\\n    bf16 = torch.cuda.is_bf16_supported()\\n    dtype = torch.bfloat16 if bf16 else torch.float16\\n    model_id = model_cfg[\\"model\\"][\\"id\\"]\\n    revision = model_cfg[\\"model\\"][\\"revision\\"]\\n    processor = AutoProcessor.from_pretrained(\\n        model_id,\\n        revision=revision,\\n        min_pixels=cfg[\\"min_pixels\\"],\\n        max_pixels=cfg[\\"max_pixels\\"],\\n    )\\n    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\\n        model_id,\\n        revision=revision,\\n        torch_dtype=dtype,\\n        device_map={\\"\\": 0},\\n        attn_implementation=\\"sdpa\\",\\n        quantization_config=BitsAndBytesConfig(\\n            load_in_4bit=True,\\n            bnb_4bit_quant_type=\\"nf4\\",\\n            bnb_4bit_use_double_quant=True,\\n            bnb_4bit_compute_dtype=dtype,\\n        ),\\n    )\\n    model = prepare_model_for_kbit_training(\\n        model,\\n        use_gradient_checkpointing=True,\\n        gradient_checkpointing_kwargs={\\"use_reentrant\\": False},\\n    )\\n    peft = model_cfg[\\"peft\\"]\\n    targets = [\\n        name\\n        for name, _ in model.named_modules()\\n        if \\"visual\\" not in name and name.rsplit(\\".\\", 1)[-1] in peft[\\"target_modules\\"]\\n    ]\\n    if not targets:\\n        raise RuntimeError(\\"No language attention projections found for LoRA\\")\\n    model = get_peft_model(\\n        model,\\n        LoraConfig(\\n            r=peft[\\"rank\\"],\\n            lora_alpha=peft[\\"alpha\\"],\\n            lora_dropout=peft[\\"dropout\\"],\\n            target_modules=targets,\\n            bias=\\"none\\",\\n            task_type=\\"CAUSAL_LM\\",\\n        ),\\n    )\\n    model.config.use_cache = False\\n    model.print_trainable_parameters()\\n    collator = VQACollator(processor, root, cfg[\\"max_length\\"])\\n    for record in records:\\n        collator([record])\\n    args.output_dir.mkdir(parents=True, exist_ok=True)\\n    git = subprocess.run([\\"git\\", \\"rev-parse\\", \\"HEAD\\"], capture_output=True, text=True)\\n    metadata = {\\n        \\"experiment_id\\": \\"M1_SFT_SMOKE\\",\\n        \\"config\\": cfg,\\n        \\"model_config\\": model_cfg,\\n        \\"git_commit\\": git.stdout.strip() if git.returncode == 0 else None,\\n        \\"annotation_sha256\\": hashlib.sha256(args.annotations.read_bytes()).hexdigest(),\\n        \\"question_ids\\": [r[\\"question_id\\"] for r in records],\\n        \\"reference_policy\\": \\"first explanation per training question\\",\\n        \\"gpu\\": torch.cuda.get_device_name(0),\\n        \\"compute_dtype\\": str(dtype),\\n        \\"versions\\": {\\n            p: importlib.metadata.version(p)\\n            for p in (\\"torch\\", \\"transformers\\", \\"peft\\", \\"accelerate\\", \\"bitsandbytes\\")\\n        },\\n        \\"status\\": \\"started\\",\\n        \\"source_sha256\\": {\\n            str(path.relative_to(Path(__file__).parents[3])): hashlib.sha256(\\n                path.read_bytes()\\n            ).hexdigest()\\n            for path in sorted(Path(__file__).parents[1].rglob(\\"*.py\\"))\\n        },\\n    }\\n    write_json(args.output_dir / \\"run_metadata.json\\", metadata)\\n\\n    class FiniteTrainer(Trainer):\\n        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):\\n            loss, outputs = super().compute_loss(\\n                model,\\n                inputs,\\n                return_outputs=True,\\n                **kwargs,\\n            )\\n            if not torch.isfinite(loss):\\n                raise FloatingPointError(\\"Non-finite SFT loss\\")\\n            return (loss, outputs) if return_outputs else loss\\n\\n    training_args = TrainingArguments(\\n        output_dir=str(args.output_dir),\\n        max_steps=cfg[\\"max_steps\\"],\\n        per_device_train_batch_size=1,\\n        gradient_accumulation_steps=cfg[\\"grad_accum\\"],\\n        learning_rate=cfg[\\"learning_rate\\"],\\n        bf16=bf16,\\n        fp16=not bf16,\\n        gradient_checkpointing=True,\\n        gradient_checkpointing_kwargs={\\"use_reentrant\\": False},\\n        logging_steps=1,\\n        logging_nan_inf_filter=False,\\n        save_steps=cfg[\\"save_steps\\"],\\n        save_total_limit=2,\\n        remove_unused_columns=False,\\n        report_to=\\"none\\",\\n        seed=cfg[\\"seed\\"],\\n        optim=\\"adamw_torch\\",\\n        dataloader_num_workers=0,\\n        max_grad_norm=1.0,\\n    )\\n    trainer = FiniteTrainer(\\n        model=model,\\n        args=training_args,\\n        train_dataset=records,\\n        data_collator=collator,\\n    )\\n    tracked = next(p for n, p in model.named_parameters() if \\"lora_B\\" in n and p.requires_grad)\\n    before = tracked.detach().float().cpu().clone()\\n    torch.cuda.reset_peak_memory_stats()\\n    result = trainer.train()\\n    delta = (tracked.detach().float().cpu() - before).abs().max().item()\\n    if not math.isfinite(delta) or delta <= 0:\\n        raise RuntimeError(\\"LoRA weights did not update\\")\\n    adapter = args.output_dir / \\"adapter\\"\\n    trainer.save_model(str(adapter))\\n    processor.save_pretrained(str(adapter))\\n    trainer.save_state()\\n    write_json(\\n        args.output_dir / \\"smoke_result.json\\",\\n        {\\n            \\"status\\": \\"passed\\",\\n            \\"global_step\\": trainer.state.global_step,\\n            \\"metrics\\": result.metrics,\\n            \\"max_lora_weight_change\\": delta,\\n            \\"peak_vram_gib\\": torch.cuda.max_memory_allocated() / 1024**3,\\n            \\"log_history\\": trainer.state.log_history,\\n            \\"note\\": \\"Training smoke only; adapter reload and validation inference still required\\",\\n        },\\n    )\\n    metadata[\\"status\\"] = \\"training_smoke_passed\\"\\n    write_json(args.output_dir / \\"run_metadata.json\\", metadata)\\n    print(f\\"SFT smoke passed; adapter: {adapter}\\")\\n","scripts/train_sft.py":"\\"\\"\\"Train the M1 answer-and-explanation adapter (8-step smoke by default).\\"\\"\\"\\n\\nfrom vqa_explanation.training.sft import main\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}'  # noqa: E501
)
for relative, content in OVERLAY.items():
    path = REPO / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
(RUN / "source_overlay.json").write_text(
    json.dumps(OVERLAY, ensure_ascii=False, indent=2), encoding="utf-8"
)
IMAGES = LOCAL / "coco"
unpack(ARCHIVE, IMAGES)
assert len(list((IMAGES / "images").glob("*.jpg"))) == 500
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(REPO),
        "transformers==4.57.1",
        "peft==0.17.1",
        "accelerate==1.10.1",
        "bitsandbytes==0.48.1",
    ],
    check=True,
)
print("READY:", RUN)

READY: /content/drive/MyDrive/Vietnamese_VQA_NLE/runs/M1_smoke_20260922_065719


In [ ]:
import os

os.chdir(REPO)
TRAIN = RUN / "training"
command = [
    sys.executable,
    "scripts/train_sft.py",
    "--annotations",
    str(ANNOTATIONS),
    "--image-root",
    str(IMAGES),
    "--output-dir",
    str(TRAIN),
]
subprocess.run(command + ["--preflight-only"], check=True)
subprocess.run(command, check=True)
print((TRAIN / "smoke_result.json").read_text())

{
  "status": "passed",
  "global_step": 8,
  "metrics": {
    "train_runtime": 64.8007,
    "train_samples_per_second": 0.247,
    "train_steps_per_second": 0.123,
    "total_flos": 118080529956864.0,
    "train_loss": 1.9063471853733063,
    "epoch": 0.5
  },
  "max_lora_weight_change": 9.030576620716602e-05,
  "peak_vram_gib": 4.322486877441406,
  "log_history": [
    {
      "loss": 2.2866,
      "grad_norm": 2.259000301361084,
      "learning_rate": 2e-05,
      "epoch": 0.0625,
      "step": 1
    },
    {
      "loss": 2.0938,
      "grad_norm": 3.531205892562866,
      "learning_rate": 1.7500000000000002e-05,
      "epoch": 0.125,
      "step": 2
    },
    {
      "loss": 2.1465,
      "grad_norm": 2.582808256149292,
      "learning_rate": 1.5000000000000002e-05,
      "epoch": 0.1875,
      "step": 3
    },
    {
      "loss": 1.6886,
      "grad_norm": 1.4021581411361694,
      "learning_rate": 1.25e-05,
      "epoch": 0.25,
      "step": 4
    },
    {
      "loss": 1.4666,

In [ ]:
PRED = RUN / "M1_smoke_validation.jsonl"
subprocess.run(
    [
        sys.executable,
        "scripts/run_m0.py",
        "--annotations",
        str(ANNOTATIONS),
        "--image-root",
        str(IMAGES),
        "--output",
        str(PRED),
        "--adapter",
        str(TRAIN / "adapter"),
        "--experiment-id",
        "M1_SFT_SMOKE",
        "--split",
        "validation",
        "--limit",
        "20",
        "--seed",
        "20260918",
        "--max-pixels",
        "200704",
        "--load-in-4bit",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/evaluate.py",
        str(PRED),
        "--output",
        str(RUN / "validation_metrics.json"),
    ],
    check=True,
)
print("ADAPTER RELOAD + INFERENCE PASSED:", RUN)
print("Chỉ smoke test: cấu hình ảnh khác M0 cũ; chưa dùng để kết luận chất lượng M1 hơn M0.")

ADAPTER RELOAD + INFERENCE PASSED: /content/drive/MyDrive/Vietnamese_VQA_NLE/runs/M1_smoke_20260922_065719
Chỉ smoke test: cấu hình ảnh khác M0 cũ; chưa dùng để kết luận chất lượng M1 hơn M0.


In [3]:
# Chạy ô khởi tạo phía trên trước; ô này không train lại.
assert Path("/content/drive/MyDrive").is_dir(), "Chạy ô khởi tạo để mount Drive"
RUN = Path("/content/drive/MyDrive/Vietnamese_VQA_NLE/runs/M1_smoke_20260922_065719")
TRAIN = RUN / "training"
PRED = RUN / "M1_smoke_validation.jsonl"
metrics = json.loads((RUN / "validation_metrics.json").read_text())
rows = [json.loads(line) for line in PRED.read_text().splitlines() if line.strip()]
assert len(rows) == 20
assert len({row["question_id"] for row in rows}) == 20
assert (TRAIN / "adapter" / "adapter_config.json").is_file()
assert (TRAIN / "adapter" / "adapter_model.safetensors").is_file()
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("VERIFIED: 20 unique predictions; adapter files present")
print("RUN:", RUN)

{
  "n": 20,
  "exact_match": 0.35,
  "token_f1": 0.4033333333333333,
  "format_valid_n": 20,
  "format_rate": 1.0,
  "latency_n": 20,
  "mean_latency_seconds": 5.8547,
  "median_latency_seconds": 5.877800000000001,
  "split": "all"
}
VERIFIED: 20 unique predictions; adapter files present
RUN: /content/drive/MyDrive/Vietnamese_VQA_NLE/runs/M1_smoke_20260922_065719


## Kết quả chạy thử

Run `M1_smoke_20260922_065719`: train 8 bước trên T4, đã lưu adapter. Suy luận 20 câu validation: EM 0,35; F1 0,4033; đúng định dạng 20/20. File kết quả ở `MyDrive/Vietnamese_VQA_NLE/runs/M1_smoke_20260922_065719/`.

Đây là mẫu kiểm tra code, chưa đủ để kết luận M1 tốt hơn M0.